# 1.3 Baseline 与瓶颈识别

## 本节学习目标

- 建立正确性 baseline
- 依据占比而非猜测选择瓶颈

## 环境检查

直接检查本节需要的运行环境；若检查失败，请先在对应 CPU/NPU 节点加载课程要求的工具链。


In [ ]:
%%bash
set -e
command -v cmake
command -v msprof
command -v npu-smi
npu-smi info
printf "ASCEND_HOME_PATH=%s\n" "${ASCEND_HOME_PATH:?请先 source CANN set_env.sh}"


## 正确性门槛

有效运行需 backend=ACL+HCCL、converged=yes、显式 relative residual<=1e-6、solution error<1e-3，且所有 rank 正常退出。

## Baseline 设计

固定矩阵、restart、tolerance、warmup、repeat、rank、线程预算和正交化方式。记录 total 及阶段计时，再按占比、调用次数和 rank 到达差异选择优化对象。

## 查看运行参数和输出

从当前章节目录执行下面的 Cell，并对照随后给出的检查点阅读输出。

## 预期现象与结果分析

残差下降但未过门槛不是有效性能结果；baseline 抖动也会扭曲 speedup。需要重复测量并报告离散程度。

## 原工程优化后历史 baseline

`src/dis_gmres/README.md` 记录了旧 Host-compute + HCCL 实现的历史结果（非当前 Device GMRES）：Ascend 910B 上六矩阵、2/4/8 rank 共 18 组优化后实验，每 rank 16 个 CPU helper thread、融合向量操作、communication-avoiding CGS，所有配置均为 `backend=ACL + HCCL`、`status=pass`。显式残差为 `3.58e-7` 到 `6.71e-7`，solution error 为 `2.67e-7` 到 `8.57e-7`。

这些记录只能作为历史参考，当前 Device GMRES 的 baseline 必须由当前真机重新运行。正式路径中 SpMV/Dot/AXPY/Norm/Scale 是 Ascend C Device kernel，HCCL 直接规约 Device buffer；计时仍需区分 kernel launch、同步、通信和首尾传输。

## 课后实践

为一个矩阵定义完整 baseline 配置和有效性检查清单。

参考答案见 `answer/01.03_answer.md`。